# Azure AI Search - Integrated Vectorization Setup

This notebook creates the complete Azure AI Search infrastructure for document indexing with vector and semantic search capabilities.

## What This Notebook Creates

1. **Blob Data Source Connector** - Connects Azure AI Search to your Azure Blob Storage
2. **Search Index** - Defines the schema with vector and semantic configurations
3. **Skillset** - Automates text splitting and embedding generation
4. **Indexer** - Orchestrates the end-to-end document processing pipeline

## Prerequisites

Before running this notebook, ensure you have:
1. An Azure AI Search service
2. An Azure Storage Account with documents uploaded (use `upload_to_blob.ipynb`)
3. An Azure OpenAI service with an embedding model deployed (text-embedding-ada-002)
4. A `.env` file with the required environment variables

## Architecture Overview

```
Blob Storage → Data Source → Indexer → Skillset → Search Index
                                         ↓
                              ┌──────────┴──────────┐
                              │                     │
                         Text Split           Embedding
                         (Chunking)          (Azure OpenAI)
```

Based on: [Azure Search Vector Samples](https://github.com/Azure/azure-search-vector-samples)

## Step 1: Import Required Libraries

We import the Azure SDK libraries for:
- **SearchIndexClient**: Managing search indexes
- **SearchIndexerClient**: Managing data sources, skillsets, and indexers
- Various model classes for configuring vector search, semantic search, skills, and projections

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    # Index models
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    
    # Data source models
    SearchIndexerDataSourceConnection,
    SearchIndexerDataContainer,
    
    # Skillset models
    SearchIndexerSkillset,
    SplitSkill,
    AzureOpenAIEmbeddingSkill,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    
    # Index projections
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    IndexProjectionMode,
    
    # Indexer models
    SearchIndexer,
    FieldMapping,
)

# Load environment variables
load_dotenv(dotenv_path="../.env")

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 2: Configure Names and Environment Variables

Define the names for all Azure AI Search components and load environment variables.

### Required Environment Variables
| Variable | Description |
|----------|-------------|
| `AZURE_AI_SERVICES_ENDPOINT` | Azure AI Search service endpoint |
| `AZURE_AI_SERVICES_KEY` | Azure AI Search admin key |
| `BLOB_CONNECTION_STRING` | Azure Storage connection string |
| `BLOB_CONTAINER_NAME` | Blob container with documents |
| `AZURE_OPENAI_ENDPOINT` | Azure OpenAI endpoint |
| `AZURE_OPENAI_KEY` | Azure OpenAI API key |
| `AZURE_OPENAI_ADA002_EMBEDDING_DEPLOYMENT` | Embedding model deployment name |

In [3]:
# Configuration - Resource Names
INDEX_NAME = "student-loan-guide-index"
DATA_SOURCE_NAME = f"{INDEX_NAME}-blob-datasource"
SKILLSET_NAME = f"{INDEX_NAME}-skillset"
INDEXER_NAME = f"{INDEX_NAME}-indexer"

print(f"📋 Resource Names:")
print(f"   - Index: {INDEX_NAME}")
print(f"   - Data Source: {DATA_SOURCE_NAME}")
print(f"   - Skillset: {SKILLSET_NAME}")
print(f"   - Indexer: {INDEXER_NAME}")

📋 Resource Names:
   - Index: student-loan-guide-index
   - Data Source: student-loan-guide-index-blob-datasource
   - Skillset: student-loan-guide-index-skillset
   - Indexer: student-loan-guide-index-indexer


In [4]:
# Get environment variables
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_AI_SERVICES_ENDPOINT")
AZURE_SEARCH_KEY = os.getenv("AZURE_AI_SERVICES_KEY")
BLOB_CONNECTION_STRING = os.getenv("BLOB_CONNECTION_STRING")
BLOB_CONTAINER_NAME = os.getenv("BLOB_CONTAINER_NAME")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_ADA002_EMBEDDING_DEPLOYMENT")

# Validate environment variables
required_vars = {
    "AZURE_AI_SERVICES_ENDPOINT": AZURE_SEARCH_ENDPOINT,
    "AZURE_AI_SERVICES_KEY": AZURE_SEARCH_KEY,
    "BLOB_CONNECTION_STRING": BLOB_CONNECTION_STRING,
    "BLOB_CONTAINER_NAME": BLOB_CONTAINER_NAME,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_KEY": AZURE_OPENAI_KEY,
    "AZURE_OPENAI_ADA002_EMBEDDING_DEPLOYMENT": AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
}

missing_vars = [var for var, value in required_vars.items() if not value]

if missing_vars:
    print("❌ Missing required environment variables:")
    for var in missing_vars:
        print(f"   - {var}")
    raise ValueError("Please set all required environment variables in .env file")
else:
    print("✅ All environment variables validated!")
    print(f"\n🔗 Azure Search Endpoint: {AZURE_SEARCH_ENDPOINT}")
    print(f"🔗 Azure OpenAI Endpoint: {AZURE_OPENAI_ENDPOINT}")
    print(f"📦 Blob Container: {BLOB_CONTAINER_NAME}")

✅ All environment variables validated!

🔗 Azure Search Endpoint: https://ai102srch193837986.search.windows.net
🔗 Azure OpenAI Endpoint: https://general-ai-projects-resource.openai.azure.com/
📦 Blob Container: content


## Step 3: Initialize Azure Search Clients

Create the client objects needed to interact with Azure AI Search:
- **SearchIndexClient**: For creating and managing indexes
- **SearchIndexerClient**: For creating data sources, skillsets, and indexers

In [5]:
# Initialize clients
credential = AzureKeyCredential(AZURE_SEARCH_KEY)
index_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential)
indexer_client = SearchIndexerClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential)

print("✅ Azure Search clients initialized!")

✅ Azure Search clients initialized!


## Step 4: Create Blob Data Source Connector

The data source connector tells Azure AI Search where to find your documents. It connects to your Azure Blob Storage container where the documents were uploaded.

**Key Configuration:**
- **Type**: `azureblob` - Connects to Azure Blob Storage
- **Connection String**: Authenticates to the storage account
- **Container**: Specifies which blob container to index

In [6]:
print("📂 Creating blob data source connector...")

container = SearchIndexerDataContainer(name=BLOB_CONTAINER_NAME)
data_source_connection = SearchIndexerDataSourceConnection(
    name=DATA_SOURCE_NAME,
    type="azureblob",
    connection_string=BLOB_CONNECTION_STRING,
    container=container
)

data_source = indexer_client.create_or_update_data_source_connection(data_source_connection)
print(f"✅ Data source '{data_source.name}' created successfully!")

📂 Creating blob data source connector...
✅ Data source 'student-loan-guide-index-blob-datasource' created successfully!


## Step 5: Create Search Index

The search index defines the schema for your searchable content. This index includes:

### Fields
| Field | Type | Purpose |
|-------|------|--------|
| `chunk_id` | String (Key) | Unique identifier for each chunk |
| `parent_id` | String | Links chunk to original document |
| `chunk` | String | The actual text content |
| `title` | String | Document title/filename |
| `vector` | Collection(Single) | 1536-dim embedding vector |

### Search Configurations
- **Vector Search**: Uses HNSW algorithm for fast approximate nearest neighbor search
- **Semantic Search**: Enables AI-powered semantic ranking of results

In [7]:
print("🔍 Creating search index...")

# Define fields for the index
fields = [
    SearchField(
        name="chunk_id",
        type=SearchFieldDataType.String,
        key=True,
        sortable=True,
        filterable=True,
        facetable=True,
        analyzer_name="keyword"
    ),
    SearchField(
        name="parent_id",
        type=SearchFieldDataType.String,
        sortable=True,
        filterable=True,
        facetable=True
    ),
    SearchField(
        name="chunk",
        type=SearchFieldDataType.String,
        sortable=False,
        filterable=False,
        searchable=True
    ),
    SearchField(
        name="title",
        type=SearchFieldDataType.String,
        searchable=True,
        filterable=True,
        sortable=True
    ),
    SearchField(
        name="vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,  # text-embedding-ada-002 dimensions
        vector_search_profile_name="vector-profile"
    ),
]

print(f"   📋 Defined {len(fields)} fields")

🔍 Creating search index...
   📋 Defined 5 fields


In [8]:
# Configure vector search with HNSW algorithm
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="hnsw-config"
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="vector-profile",
            algorithm_configuration_name="hnsw-config"
        )
    ]
)

print("   🧮 Vector search configured with HNSW algorithm")

   🧮 Vector search configured with HNSW algorithm


In [9]:
# Configure semantic search
semantic_config = SemanticConfiguration(
    name="semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        content_fields=[SemanticField(field_name="chunk")]
    )
)

semantic_search = SemanticSearch(configurations=[semantic_config])

print("   🧠 Semantic search configured")

   🧠 Semantic search configured


In [10]:
# Create the search index
index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

result = index_client.create_or_update_index(index)
print(f"✅ Index '{result.name}' created successfully!")

✅ Index 'student-loan-guide-index' created successfully!


## Step 6: Create Skillset

The skillset defines the AI enrichment pipeline that processes documents:

### Skills in this Pipeline

1. **Text Split Skill**
   - Chunks documents into smaller pieces (max 2000 characters)
   - Uses 500 character overlap for context continuity
   
2. **Azure OpenAI Embedding Skill**
   - Generates 1536-dimension vectors using text-embedding-ada-002
   - Runs on each chunk produced by the split skill

### Index Projections
Maps the chunked and embedded content to the search index fields.

In [11]:
print("🛠️  Creating skillset...")

# Text Split Skill - chunks the document
split_skill = SplitSkill(
    description="Split skill to chunk documents",
    text_split_mode="pages",
    context="/document",
    maximum_page_length=2000,
    page_overlap_length=500,
    inputs=[
        InputFieldMappingEntry(name="text", source="/document/content")
    ],
    outputs=[
        OutputFieldMappingEntry(name="textItems", target_name="pages")
    ]
)

print("   ✂️  Text Split Skill configured (max 2000 chars, 500 overlap)")

🛠️  Creating skillset...
   ✂️  Text Split Skill configured (max 2000 chars, 500 overlap)


In [12]:
# Azure OpenAI Embedding Skill - generates embeddings
embedding_skill = AzureOpenAIEmbeddingSkill(
    description="Skill to generate embeddings via Azure OpenAI",
    context="/document/pages/*",
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    model_name="text-embedding-ada-002",
    api_key=AZURE_OPENAI_KEY,
    inputs=[
        InputFieldMappingEntry(name="text", source="/document/pages/*")
    ],
    outputs=[
        OutputFieldMappingEntry(name="embedding", target_name="vector")
    ]
)

print(f"   🔢 Embedding Skill configured (model: text-embedding-ada-002)")

   🔢 Embedding Skill configured (model: text-embedding-ada-002)


In [13]:
# Index projections - maps chunks to the index
index_projections = SearchIndexerIndexProjection(
    selectors=[
        SearchIndexerIndexProjectionSelector(
            target_index_name=INDEX_NAME,
            parent_key_field_name="parent_id",
            source_context="/document/pages/*",
            mappings=[
                InputFieldMappingEntry(name="chunk", source="/document/pages/*"),
                InputFieldMappingEntry(name="vector", source="/document/pages/*/vector"),
                InputFieldMappingEntry(name="title", source="/document/metadata_storage_name")
            ]
        )
    ],
    parameters=SearchIndexerIndexProjectionsParameters(
        projection_mode=IndexProjectionMode.SKIP_INDEXING_PARENT_DOCUMENTS
    )
)

print("   📊 Index projections configured")

   📊 Index projections configured


In [14]:
# Create the skillset with index projections
skillset = SearchIndexerSkillset(
    name=SKILLSET_NAME,
    description="Skillset to chunk documents and generate embeddings",
    skills=[split_skill, embedding_skill],
    index_projection=index_projections
)

result = indexer_client.create_or_update_skillset(skillset)
print(f"✅ Skillset '{result.name}' created successfully!")

✅ Skillset 'student-loan-guide-index-skillset' created successfully!


## Step 7: Create and Run Indexer

The indexer orchestrates the entire document processing pipeline:
1. Reads documents from the blob data source
2. Applies the skillset (chunking + embedding)
3. Populates the search index with the results

**Note:** The indexer runs asynchronously. It may take several minutes to process all documents depending on volume.

In [15]:
print("⚙️  Creating indexer...")

indexer = SearchIndexer(
    name=INDEXER_NAME,
    description="Indexer to process documents and generate embeddings",
    skillset_name=SKILLSET_NAME,
    target_index_name=INDEX_NAME,
    data_source_name=DATA_SOURCE_NAME,
    field_mappings=[
        FieldMapping(
            source_field_name="metadata_storage_name",
            target_field_name="title"
        )
    ]
)

result = indexer_client.create_or_update_indexer(indexer)
print(f"✅ Indexer '{result.name}' created successfully!")

⚙️  Creating indexer...
✅ Indexer 'student-loan-guide-index-indexer' created successfully!


In [ ]:
# Run the indexer
print(f"\n🚀 Running indexer '{INDEXER_NAME}'...")
indexer_client.run_indexer(INDEXER_NAME)
print("✅ Indexer is now running!")
print("\n⏳ Note: It may take a few minutes to complete processing.")
print("   You can check the status in the next cell or in Azure Portal.")

## Step 8: Check Indexer Status

Monitor the indexer to see when processing is complete. Run this cell periodically to check progress.

In [18]:
print(f"📊 Checking indexer status...")

try:
    status = indexer_client.get_indexer_status(INDEXER_NAME)
    print(f"\nIndexer Status: {status.status}")
    print(f"Last Result: {status.last_result.status if status.last_result else 'N/A'}")
    
    if status.last_result:
        print(f"Items Processed: {status.last_result.items_processed}")
        print(f"Items Failed: {status.last_result.items_failed}")
        
        if status.last_result.errors:
            print("\n❌ Errors:")
            for error in status.last_result.errors:
                print(f"   - {error.error_message}")
        else:
            print("\n✅ No errors!")
            
except Exception as e:
    print(f"Could not retrieve status: {str(e)}")

📊 Checking indexer status...

Indexer Status: running
Last Result: success
Could not retrieve status: 'IndexerExecutionResult' object has no attribute 'items_processed'


## Summary

You have successfully created the complete Azure AI Search infrastructure!

In [19]:
print("=" * 80)
print("✅ Setup Complete!")
print("=" * 80)
print(f"\n📋 Summary:")
print(f"   - Index Name: {INDEX_NAME}")
print(f"   - Data Source: {DATA_SOURCE_NAME}")
print(f"   - Skillset: {SKILLSET_NAME}")
print(f"   - Indexer: {INDEXER_NAME}")
print(f"\n💡 Next steps:")
print(f"   1. Wait for the indexer to complete processing")
print(f"   2. Check indexer status in Azure Portal or re-run the status cell above")
print(f"   3. Run search_queries.ipynb to test vector and hybrid searches")

✅ Setup Complete!

📋 Summary:
   - Index Name: student-loan-guide-index
   - Data Source: student-loan-guide-index-blob-datasource
   - Skillset: student-loan-guide-index-skillset
   - Indexer: student-loan-guide-index-indexer

💡 Next steps:
   1. Wait for the indexer to complete processing
   2. Check indexer status in Azure Portal or re-run the status cell above
   3. Run search_queries.ipynb to test vector and hybrid searches
